In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from pyampute.exploration.mcar_statistical_tests import MCARTest
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, FunctionTransformer, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split, GroupKFold, GridSearchCV
from sklearn.metrics import r2_score

In [2]:
df = pd.read_csv("data/Life Expectancy Data.csv")
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_").str.replace('/', '_').str.replace('-', '_')
df.head()

,country,year,status,life_expectancy,adult_mortality,infant_deaths,alcohol,percentage_expenditure,hepatitis_b,measles,...,polio,total_expenditure,diphtheria,hiv_aids,gdp,population,thinness__1_19_years,thinness_5_9_years,income_composition_of_resources,schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,...,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,...,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5


In [4]:
df = df.dropna(subset=['life_expectancy'])
df.describe()

,year,life_expectancy,adult_mortality,infant_deaths,alcohol,percentage_expenditure,hepatitis_b,measles,bmi,under_five_deaths,polio,total_expenditure,diphtheria,hiv_aids,gdp,population,thinness__1_19_years,thinness_5_9_years,income_composition_of_resources,schooling
count,2928.00000,2928.000000,2928.000000,2928.000000,2735.000000,2928.000000,2375.000000,2928.000000,2896.000000,2928.000000,2909.000000,2702.000000,2909.000000,2928.000000,2485.000000,2.284000e+03,2896.000000,2896.000000,2768.000000,2768.000000
mean,2007.50000,69.224932,164.796448,30.407445,4.614856,740.321185,80.960842,2427.855874,38.235394,42.179303,82.548298,5.930163,82.321416,1.747712,7494.210719,1.276454e+07,4.850622,4.881423,0.627419,11.999639
std,4.61056,9.523867,124.292079,118.114450,4.050749,1990.930605,25.018337,11485.970937,19.959590,160.700547,23.416674,2.483273,23.706644,5.085542,14282.251492,6.103765e+07,4.420829,4.509609,0.210978,3.346440
min,2000.00000,36.300000,1.000000,0.000000,0.010000,0.000000,1.000000,0.000000,1.000000,0.000000,3.000000,0.370000,2.000000,0.100000,1.681350,3.400000e+01,0.100000,0.100000,0.000000,0.000000
25%,2003.75000,63.100000,74.000000,0.000000,0.905000,4.853964,77.000000,0.000000,19.300000,0.000000,78.000000,4.260000,78.000000,0.100000,463.852618,1.966738e+05,1.600000,1.575000,0.493000,10.100000
50%,2007.50000,72.100000,144.000000,3.000000,3.770000,65.611455,92.000000,17.000000,43.350000,4.000000,93.000000,5.750000,93.000000,0.100000,1764.973870,1.391756e+06,3.300000,3.400000,0.677000,12.300000
75%,2011.25000,75.700000,228.000000,22.000000,7.715000,442.614322,97.000000,362.250000,56.100000,28.000000,97.000000,7.490000,97.000000,0.800000,5932.899677,7.426746e+06,7.200000,7.200000,0.779250,14.300000
max,2015.00000,89.000000,723.000000,1800.000000,17.870000,19479.911610,99.000000,212183.000000,77.600000,2500.000000,99.000000,17.600000,99.000000,50.600000,119172.741800,1.293859e+09,27.700000,28.600000,0.948000,20.700000


In [23]:
def check_missingness(df: pd.DataFrame) -> pd.DataFrame:

    treated_df = df.copy()

    encoded_df = df.copy()


    # Temparory Encoding 
    label_encoders = {}
    for col in encoded_df.select_dtypes(include=["object", "str"]).columns.tolist():
        label_encoders[col] = LabelEncoder()
        
        encoded_df[col] = label_encoders[col].fit_transform(encoded_df[col])
    

    # Identify the Features with no Null values 
    missing_percentages = df.isnull().mean()

    complete_features = missing_percentages[missing_percentages == 0].index.tolist()

    # If No column are 100% complete pick columns with <1% missingness
    if len(complete_features) < 2:
        complete_features = missing_percentages[missing_percentages < 0.01].index.tolist()

        encoded_df = encoded_df.dropna(subset = complete_features)
    
    # print("Running MCAR Test....")
    mcar_test = MCARTest()
    p_value = mcar_test(df.select_dtypes(exclude=["object", "str"]))
    # print(f"p value : {p_value}")

    # If p_value > 0.05 means Null Hypothesis is Correct Data is MCAR
    if p_value > 0.05: 
        # Treat as MCAR
        # print("Data is MCAR")
        num_cols = treated_df.select_dtypes(exclude=["object", "str"]).columns.tolist()

        for col in num_cols:
            if treated_df[col].isnull().any():
                treated_df[col] = treated_df[col].fillna(treated_df[col].median())
        return treated_df

    # If p_value < 0.05 
    # print("Data is Not MCAR")
    # print("Checking btw MAR and MNAR")

    columns_to_test = missing_percentages[missing_percentages > 0].index.tolist()
    mar_cols = []
    mnar_cols = []

    # Removing the country columns because lower gdp and population countries will never give there name so my model don't learn it, it memorizes that if this country came the value will be missing 
    complete_features.remove('country')

    # Checking each row individually
    for col in columns_to_test:

        pct = missing_percentages[col]*100

        # if missing values are <1% then just fill it with mean or median
        if pct < 1:
            treated_df[col] = treated_df[col].fillna(treated_df[col].median())
            continue

        # creating a missing indicator for model
        encoded_df['is_missing'] = encoded_df[col].isnull().astype(int) 
        
        X = encoded_df[complete_features]
        y = encoded_df['is_missing']

        clf = RandomForestClassifier(n_estimators=100, random_state=0)

        scores = cross_val_score(clf, X, y, cv = 5, scoring='roc_auc')

        mean_auc = scores.mean()
        # print(f"AUC for {col} : {mean_auc}")
        # If mean_auc >= 0.70 the columns has high relation and will be added to mar_cols
        if mean_auc >= 0.70:
            mar_cols.append(col)
        # if not then it is suspected to be MNAR and need to add missing indicator
        else:
            treated_df[f"{col}_was_missing"] = treated_df[col].isnull().astype(int)
            mnar_cols.append(col)
    # print(f"MAR Columns : {mar_cols}")
    # print(f"MNAR Columns : {mnar_cols}")
    
    # Imputation for MAR/MNAR columns
    if mar_cols or mnar_cols:
        num_cols = treated_df.select_dtypes(exclude=['object', 'str']).columns.tolist()

        mice = IterativeImputer(max_iter=50, random_state=0, min_value=0)

        treated_df[num_cols] = mice.fit_transform(treated_df[num_cols])
    # print("Data Imputation is complete")
    return treated_df.drop(columns=['country'])  

<h4> Since Lasso Regressor has automatic feature selection we don't need to do that</h4>

In [24]:
imputer = FunctionTransformer(func=check_missingness)

status_encoder_pipe = Pipeline(steps = [
    ('encode', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

scaler_pipe = Pipeline(steps = [
    ('scaler', StandardScaler())
])

positive_skewness_pipe = Pipeline(steps = [
    ('log', FunctionTransformer(func=np.log1p, feature_names_out='one-to-one')),
    ('scaler', StandardScaler())
])

negative_skewness_pipe = Pipeline(steps = [
    ('square', FunctionTransformer(func=np.square, feature_names_out='one-to-one')),
    ('scaler', StandardScaler())
])

heavy_skewness_pipe = Pipeline(steps = [
    ('yeo-johnson', PowerTransformer(method='yeo-johnson'))
])

In [25]:
imputed_df = check_missingness(df)

In [9]:
skewness = imputed_df.skew(numeric_only=True).sort_values(ascending=False)
skewness

population                         17.872871
infant_deaths                       9.771044
under_five_deaths                   9.479623
measles                             9.425290
hiv_aids                            5.386623
percentage_expenditure              4.643790
gdp                                 3.525832
thinness_5_9_years                  1.775036
thinness__1_19_years                1.709896
adult_mortality                     1.174369
alcohol                             0.624825
total_expenditure                   0.605146
year                                0.000000
bmi                                -0.210141
schooling                          -0.552553
life_expectancy                    -0.638605
income_composition_of_resources    -1.094580
hepatitis_b                        -1.634918
diphtheria                         -2.083450
polio                              -2.108851
dtype: float64

In [10]:
positive_skewness_cols = skewness[(skewness>0.5) & (skewness<1)].index.tolist()
negative_skewness_cols = skewness[(skewness<-0.5) & (skewness>-1)].index.tolist()
negative_skewness_cols.remove('life_expectancy')
heavy_skewness_cols = skewness[(skewness > 1) | (skewness < -1)].index.tolist()

In [11]:
transformer = ColumnTransformer(transformers=[
    ('status_encoder', status_encoder_pipe, ['status']),
    ('positive_skewness', positive_skewness_pipe, positive_skewness_cols),
    ('negative_skewness', negative_skewness_pipe, negative_skewness_cols),
    ('heavy_skewness', heavy_skewness_pipe, heavy_skewness_cols),
    ("scaler", scaler_pipe, ['year', 'bmi'])
], remainder='passthrough', verbose_feature_names_out=False)

In [12]:
transformer.fit(imputed_df)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('status_encoder', ...), ('positive_skewness', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transforme

In [13]:
transformed_df = pd.DataFrame(transformer.transform(imputed_df),
                              columns=transformer.get_feature_names_out())
transformed_df.skew()

status_Developing                 -1.712798
alcohol                           -0.292412
total_expenditure                 -0.674318
schooling                          0.378070
population                        -0.000667
infant_deaths                      0.173319
under_five_deaths                  0.165568
measles                            0.178441
hiv_aids                           0.965566
percentage_expenditure            -0.014672
gdp                                0.017344
thinness_5_9_years                 0.012814
thinness__1_19_years               0.017912
adult_mortality                   -0.080304
income_composition_of_resources   -0.149334
hepatitis_b                       -0.854334
diphtheria                        -1.043284
polio                             -1.027324
year                               0.000000
bmi                               -0.210141
life_expectancy                   -0.638605
dtype: float64

In [27]:
final_model = Pipeline(steps = [
    ('imputation', imputer),
    ('transformer', transformer),
    ('lasso', Lasso())
])

In [15]:
X = df.drop(columns=['life_expectancy'])
y = df['life_expectancy']

X_train, X_test, y_train, y_test = train_test_split(X, y ,test_size = 0.3)

In [28]:
final_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputation', ...), ('transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function che...00223F9C5F420>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword arguments

In [29]:
pred = final_model.predict(X_test)

r2_score(y_test, pred)

0.8202406649077871

In [30]:
params = {'lasso__alpha' : [0.001, 0.01, 0.1, 1, 2, 5]}

group_kfolds = GroupKFold(n_splits=5)

grid_search = GridSearchCV(
    estimator=final_model,
    param_grid=params,
    scoring='r2',
    cv=group_kfolds
)

grid_search.fit(
    X_train, 
    y_train, 
    groups=df.loc[X_train.index, 'country']
)

grid_search.best_params_

{'lasso__alpha': 0.1}